# PhoBERT Baseline - Cross Entropy Loss

## Mô tả
Notebook này triển khai mô hình PhoBERT baseline cho phân tích cảm xúc sinh viên.

**Đặc điểm:**
- Sử dụng PhoBERT pre-trained (vinai/phobert-base)
- Loss function: CrossEntropyLoss (standard)
- Không sử dụng data augmentation
- Không sử dụng class weights

**Mục đích:** Làm baseline để so sánh với các phương pháp cải tiến.

## 1. Setup và Import Libraries

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install transformers torch scikit-learn matplotlib seaborn

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2. Configuration

In [ ]:
class Config:
    BASE_DIR = '/content/drive/MyDrive/Student-Feedback-Sentiment-Analysis'
    DATA_DIR = f'{BASE_DIR}/data/processed'
    # NEW: Hierarchical results directory structure
    # Format: results/{ModelType}/{baseline|improvements}/
    MODEL_TYPE = 'PhoBERT'
    EXPERIMENT_TYPE = 'baseline'  # Baseline has no timestamp
    RESULTS_DIR = f'{BASE_DIR}/results/{MODEL_TYPE}/{EXPERIMENT_TYPE}'
    
    MODEL_NAME = 'vinai/phobert-base'
    MAX_LENGTH = 256
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 10
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    EARLY_STOPPING_PATIENCE = 5
    NUM_CLASSES = 3
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Subdirectories for organized outputs
    MODELS_DIR = os.path.join(RESULTS_DIR, 'models')
    SUMMARIES_DIR = os.path.join(RESULTS_DIR, 'summaries')
    VISUALIZATIONS_DIR = os.path.join(RESULTS_DIR, 'visualizations')
    ARTIFACTS_DIR = os.path.join(RESULTS_DIR, 'artifacts')

config = Config()
# Create all subdirectories
for dir_path in [config.RESULTS_DIR, config.MODELS_DIR, config.SUMMARIES_DIR, 
                 config.VISUALIZATIONS_DIR, config.ARTIFACTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    
print(f'Results will be saved to: {config.RESULTS_DIR}')
print(f'Device: {config.DEVICE}')
print(f'Structure: models/, summaries/, visualizations/, artifacts/')

## 3. Load Data

In [7]:
def load_data(data_dir, split):
    split_dir = os.path.join(data_dir, split)
    with open(os.path.join(split_dir, 'sents.txt'), 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f.readlines()]
    with open(os.path.join(split_dir, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f'{split}: {len(texts)} samples')
    return texts, labels

train_texts, train_labels = load_data(config.DATA_DIR, 'train')
val_texts, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts, test_labels = load_data(config.DATA_DIR, 'test')

train: 11426 samples
validation: 1583 samples
test: 3166 samples


In [8]:
from collections import Counter

print('Label distribution:')
for split_name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    counter = Counter(labels)
    print(f'{split_name}:')
    for label, count in sorted(counter.items()):
        print(f'  {config.LABEL_MAP[label]}: {count} ({count/len(labels)*100:.1f}%)')

Label distribution:
Train:
  Negative: 5325 (46.6%)
  Neutral: 458 (4.0%)
  Positive: 5643 (49.4%)
Val:
  Negative: 705 (44.5%)
  Neutral: 73 (4.6%)
  Positive: 805 (50.9%)
Test:
  Negative: 1409 (44.5%)
  Neutral: 167 (5.3%)
  Positive: 1590 (50.2%)


## 4. Dataset và DataLoader

In [9]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f'Tokenizer loaded: {config.MODEL_NAME}')

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9cd8ac62-b0a9-4330-9a3d-b57f2afbe4cc)')' thrown while requesting HEAD https://huggingface.co/vinai/phobert-base/resolve/main/chat_template.jinja
Retrying in 1s [Retry 1/5].


Tokenizer loaded: vinai/phobert-base


In [10]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [11]:
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, config.MAX_LENGTH)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, config.MAX_LENGTH)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, config.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 715
Val batches: 99
Test batches: 198


## 5. Model Definition

In [12]:
class PhoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [13]:
model = PhoBERTClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
model = model.to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Total parameters: 135,000,579
Trainable parameters: 135,000,579


## 6. Training Setup - CrossEntropyLoss (Baseline)

In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print('Loss function: CrossEntropyLoss (standard)')
print(f'Total training steps: {total_steps}')
print(f'Warmup steps: {warmup_steps}')

Loss function: CrossEntropyLoss (standard)
Total training steps: 7150
Warmup steps: 715


## 7. Training Functions

In [15]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, accuracy, f1

In [16]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, accuracy, f1, all_preds, all_labels

## 8. Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'train_f1': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0
patience_counter = 0

print('='*60)
print('Starting Training - PhoBERT Baseline (CrossEntropyLoss)')
print('='*60)

for epoch in range(config.NUM_EPOCHS):
    print(f'Epoch {epoch + 1}/{config.NUM_EPOCHS}')
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, scheduler, config.DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, config.DEVICE)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    print(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        # NEW: Save to models directory
        torch.save(model.state_dict(), os.path.join(config.MODELS_DIR, 'phobert_model.pt'))
        print(f'New best model saved! Val F1: {val_f1:.4f}')
    else:
        patience_counter += 1
        print(f'No improvement. Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}')
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f'Early stopping triggered at epoch {epoch + 1}')
            break

print('='*60)
print(f'Training completed! Best Val F1: {best_val_f1:.4f}')
print('='*60)

## 9. Training Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)
axes[2].plot(history['train_f1'], label='Train')
axes[2].plot(history['val_f1'], label='Validation')
axes[2].set_title('F1 Score over Epochs')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].legend()
axes[2].grid(True)
plt.tight_layout()
# NEW: Save to visualizations directory
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'training_history.png'), dpi=150)
plt.show()

## 10. Evaluation on Test Set

In [ ]:
# NEW: Load from models directory
model.load_state_dict(torch.load(os.path.join(config.MODELS_DIR, 'phobert_model.pt')))
print('Best model loaded for evaluation')

test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, criterion, config.DEVICE)

print('='*60)
print('TEST SET RESULTS - PhoBERT Baseline (CrossEntropyLoss)')
print('='*60)
print(f'Loss: {test_loss:.4f}')
print(f'Accuracy: {test_acc:.4f}')
print(f'F1 Score (weighted): {test_f1:.4f}')

In [20]:
print('Classification Report:')
target_names = [config.LABEL_MAP[i] for i in range(config.NUM_CLASSES)]
print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

Classification Report:
              precision    recall  f1-score   support

    Negative     0.9441    0.9588    0.9514      1409
     Neutral     0.6935    0.5150    0.5911       167
    Positive     0.9429    0.9553    0.9491      1590

    accuracy                         0.9337      3166
   macro avg     0.8602    0.8097    0.8305      3166
weighted avg     0.9303    0.9337    0.9312      3166



In [ ]:
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - PhoBERT Baseline')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
# NEW: Save to visualizations directory
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 11. Save Results

In [ ]:
import pandas as pd

summary = {
    'Model': 'PhoBERT Baseline',
    'Loss Function': 'CrossEntropyLoss',
    'Data Augmentation': 'No',
    'Class Weights': 'No',
    'Epochs Trained': len(history['train_loss']),
    'Best Val F1': best_val_f1,
    'Test Accuracy': test_acc,
    'Test F1 (weighted)': test_f1,
    'Test Loss': test_loss
}

summary_df = pd.DataFrame([summary])
# NEW: Save to summaries directory
summary_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'summary.csv'), index=False)

print('='*60)
print('💾 SAVING RESULTS')
print('='*60)
print('Results Summary:')
for key, value in summary.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')

print(f'\n📁 All results saved to: {config.RESULTS_DIR}')
print(f'   ├── models/phobert_model.pt')
print(f'   ├── summaries/summary.csv')
print(f'   └── visualizations/training_history.png, confusion_matrix.png')

## 12. Summary

### PhoBERT Baseline Model

**Configuration:**
- Model: vinai/phobert-base
- Loss: CrossEntropyLoss (standard)
- No data augmentation
- No class weights
- Epochs: 10 (with early stopping)

**Purpose:**
This baseline model serves as a reference point for comparing with enhanced versions like:
- PhoBERT + Focal Loss
- PhoBERT + Data Augmentation
- PhoBERT + Class Weights